In [1]:
!pip install -q groq

from groq import Groq
client = Groq(api_key="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")  # paste new key after regenerating
models = client.models.list()
for m in models.data:
    print(m.id)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.2 MB/s eta 0:00:00
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo
openai/gpt-oss-120b
llama-3.3-70b-versatile
qwen/qwen3-32b
qwen/qwen3.6-27b
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-safeguard-20b
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
llama-3.1-8b-instant
meta-llama/llama-4-scout-17b-16e-instruct
groq/compound-mini
whisper-large-v3
canopylabs/orpheus-v1-english
groq/compound
openai/gpt-oss-20b


In [2]:
# CELL 1: Install + Mount Drive
!pip install groq --quiet
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# CELL 2: Config — SET YOUR VALUES HERE
import os

GROQ_API_KEYS = [
     os.environ.get("GROQ_API_KEY_1", "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"),  # Account 1
     os.environ.get("GROQ_API_KEY_2", "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"),  # Account 2
     os.environ.get("GROQ_API_KEY_3", "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"),  # Account 3
     os.environ.get("GROQ_API_KEY_4", "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"),  # Account 4

]
GROQ_API_KEYS = [k for k in GROQ_API_KEYS if k and not k.startswith("your-key")]
assert len(GROQ_API_KEYS) >= 1, "Add at least one Groq API key!"

# Confirmed available on Groq as of June 2026
MODELS_TO_EVALUATE = [
    "llama-3.1-8b-instant",                        # Primary baseline (used in CP-1/CP-2)
    "llama-3.3-70b-versatile",                      # Larger Llama variant
    "meta-llama/llama-4-scout-17b-16e-instruct",    # Llama 4 family
    "qwen/qwen3-32b",                               # Different architecture (Alibaba)
]
# NOTE: mixtral-8x7b-32768 and gemma2-9b-it are NO LONGER available — do not use.

DATA_DIR = "/content/rgb_data"
os.makedirs(DATA_DIR, exist_ok=True)

SAVE_PATH = "/content/drive/MyDrive/RAGBench_Results/CP4_RGB"
os.makedirs(SAVE_PATH, exist_ok=True)

QUICK_TEST = False      # Set True to run only 3 examples while debugging
MAX_EXAMPLES = 3 if QUICK_TEST else None
NOISE_RATIOS = [0.0, 0.2, 0.4, 0.6, 0.8]
CHECKPOINT_EVERY = 5

print(f"Config loaded. Models: {MODELS_TO_EVALUATE}")
print(f"QUICK_TEST={QUICK_TEST}, MAX_EXAMPLES={MAX_EXAMPLES}")
print(f"Save path: {SAVE_PATH}")

Config loaded. Models: ['llama-3.1-8b-instant', 'llama-3.3-70b-versatile', 'meta-llama/llama-4-scout-17b-16e-instruct', 'qwen/qwen3-32b']
QUICK_TEST=False, MAX_EXAMPLES=None
Save path: /content/drive/MyDrive/RAGBench_Results/CP4_RGB


In [4]:
# CELL 3: Verify Groq Model Availability
# Models are confirmed available as of June 2026.
# This cell does a quick sanity-check and will warn if anything has changed.
from groq import Groq

client_check = Groq(api_key=GROQ_API_KEYS[0])
available_models = [m.id for m in client_check.models.list().data]

print("--- Checking confirmed model list ---")
valid_models = []
for m in MODELS_TO_EVALUATE:
    if m in available_models:
        print(f"  OK  {m}")
        valid_models.append(m)
    else:
        print(f"  WARN {m} not found — check model ID or Groq account")

MODELS_TO_EVALUATE = valid_models
assert len(MODELS_TO_EVALUATE) >= 1, "No valid models found! Check your Groq API key."
print(f"\nFinal model list ({len(MODELS_TO_EVALUATE)} models): {MODELS_TO_EVALUATE}")

--- Checking confirmed model list ---
  OK  llama-3.1-8b-instant
  OK  llama-3.3-70b-versatile
  OK  meta-llama/llama-4-scout-17b-16e-instruct
  OK  qwen/qwen3-32b

Final model list (4 models): ['llama-3.1-8b-instant', 'llama-3.3-70b-versatile', 'meta-llama/llama-4-scout-17b-16e-instruct', 'qwen/qwen3-32b']


In [5]:
# CELL 4: Download RGB Dataset from GitHub
import urllib.request

RGB_FILES = {
    "en_refine": "en_refine.json",
    "en_int":    "en_int.json",
    "en_fact":   "en_fact.json",
}
BASE_URL = "https://raw.githubusercontent.com/chen700564/RGB/master/data/"

for key, fname in RGB_FILES.items():
    local_path = os.path.join(DATA_DIR, fname)
    if os.path.exists(local_path):
        print(f"  Already exists: {fname}")
    else:
        url = BASE_URL + fname
        print(f"  Downloading {fname}...")
        urllib.request.urlretrieve(url, local_path)
        print(f"  Saved to {local_path}")

print("All RGB files ready.")

  Saved to /content/rgb_data/en_refine.json
  Saved to /content/rgb_data/en_int.json
  Saved to /content/rgb_data/en_fact.json
All RGB files ready.


In [6]:
# CELL 5: Data Loader + Testbed Builder
import json, random

def load_jsonl(filepath):
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def build_noise_testbed(records, noise_ratio, max_examples=None, seed=42):
    random.seed(seed)
    testbed = []
    recs = records[:max_examples] if max_examples else records
    for rec in recs:
        positives = rec.get("positive", [])
        negatives = rec.get("negative", [])
        if not positives:
            continue
        n_docs = 5
        n_noise = round(n_docs * noise_ratio)
        n_pos = n_docs - n_noise
        pos_sample = random.sample(positives, min(n_pos, len(positives)))
        neg_sample = random.sample(negatives, min(n_noise, len(negatives))) if negatives else []
        while len(pos_sample) < n_pos and positives:
            pos_sample.append(random.choice(positives))
        docs = pos_sample + neg_sample
        random.shuffle(docs)
        testbed.append({"id": rec["id"], "query": rec["query"], "answer": rec["answer"],
                        "docs": docs, "noise_ratio": noise_ratio, "ability": "noise_robustness"})
    return testbed


def build_negative_rejection_testbed(records, max_examples=None, seed=42):
    random.seed(seed)
    testbed = []
    recs = records[:max_examples] if max_examples else records
    for rec in recs:
        negatives = rec.get("negative", [])
        if not negatives:
            continue
        n_docs = 5
        neg_sample = random.sample(negatives, min(n_docs, len(negatives)))
        while len(neg_sample) < n_docs and negatives:
            neg_sample.append(random.choice(negatives))
        testbed.append({"id": rec["id"], "query": rec["query"], "answer": rec["answer"],
                        "docs": neg_sample, "ability": "negative_rejection"})
    return testbed


def build_info_integration_testbed(records, max_examples=None):
    recs = records[:max_examples] if max_examples else records
    testbed = []
    for rec in recs:
        positives = rec.get("positive", [])
        negatives = rec.get("negative", [])
        docs = positives[:5]
        if len(docs) < 5 and negatives:
            docs += negatives[:5 - len(docs)]
        testbed.append({"id": rec["id"], "query": rec["query"], "answer": rec["answer"],
                        "docs": docs, "ability": "info_integration"})
    return testbed


def build_counterfactual_testbed(records, max_examples=None):
    recs = records[:max_examples] if max_examples else records
    testbed = []
    for rec in recs:
        docs = rec.get("positive", [])[:5]
        testbed.append({"id": rec["id"], "query": rec["query"], "answer": rec["answer"],
                        "docs": docs, "ability": "counterfactual_robustness"})
    return testbed


en_refine = load_jsonl(os.path.join(DATA_DIR, "en_refine.json"))
en_int    = load_jsonl(os.path.join(DATA_DIR, "en_int.json"))
en_fact   = load_jsonl(os.path.join(DATA_DIR, "en_fact.json"))

print(f"Loaded: en_refine={len(en_refine)}, en_int={len(en_int)}, en_fact={len(en_fact)}")
print(f"Sample keys: {list(en_refine[0].keys())}")
print(f"Sample answer: {en_refine[0]['answer']}")

Loaded: en_refine=300, en_int=100, en_fact=100
Sample keys: ['id', 'query', 'answer', 'positive', 'negative']
Sample answer: [['January 2 2022', 'Jan 2, 2022', 'Jan. 2, 2022', 'January 2, 2022', '2 January 2022', '2 Jan, 2022', '2 Jan., 2022', '2 January, 2022']]


In [7]:
# CELL 6: Prompt Templates (based on RGB paper Figure 3)
SYSTEM_PROMPT_BASE = (
    "You are a helpful assistant that answers questions based on provided documents. "
    "Answer using only the information in the documents. If the documents do not contain "
    "the answer, output exactly: I can not answer the question because of the insufficient "
    "information in documents."
)

SYSTEM_PROMPT_COUNTERFACTUAL = (
    "You are a helpful assistant that answers questions based on provided documents. "
    "Note: The provided documents may contain factual errors. "
    "If you detect factual errors, output exactly: There are factual errors in the provided documents. "
    "Then provide the correct answer based on your knowledge."
)


def format_docs(docs):
    parts = []
    for i, doc in enumerate(docs, 1):
        text = doc[:1500] if len(doc) > 1500 else doc  # Prevent 413 errors
        parts.append(f"Document {i}: {text}")
    return "\n\n".join(parts)


def build_user_prompt(query, docs, ability):
    doc_text = format_docs(docs)
    if ability == "counterfactual_robustness":
        return (
            f"The following documents may contain factual errors. "
            f"If you detect factual errors, say: There are factual errors in the provided documents. "
            f"Then provide the correct answer.\n\n{doc_text}\n\nQuestion: {query}\nAnswer:"
        )
    return f"{doc_text}\n\nQuestion: {query}\nAnswer:"


def get_system_prompt(ability):
    return SYSTEM_PROMPT_COUNTERFACTUAL if ability == "counterfactual_robustness" else SYSTEM_PROMPT_BASE


print("Prompt templates ready.")

Prompt templates ready.


In [8]:
# CELL 7: Groq Multi-Key Rotation + Retry
import time, itertools

_key_cycle = itertools.cycle(GROQ_API_KEYS)
_groq_clients = {k: Groq(api_key=k) for k in GROQ_API_KEYS}


def call_groq_with_retry(model, system_prompt, user_prompt, max_retries=5):
    last_error = None
    for attempt in range(max_retries):
        key = next(_key_cycle)
        client = _groq_clients[key]
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": system_prompt},
                           {"role": "user",   "content": user_prompt}],
                max_tokens=512,
                temperature=0.0,
            )
            #time.sleep(2)   # ← ADD THIS — 2s base delay between every call
            # Larger models consume more tokens — need longer breathing room
            sleep_time = 10 if any(x in model.lower() for x in ["70b", "scout", "qwen3-32b"]) else 2
            time.sleep(sleep_time)
            return resp.choices[0].message.content.strip()
        except Exception as e:
            last_error = e
            err_str = str(e).lower()
            if "rate_limit" in err_str or "429" in err_str:
                wait = 2 ** (attempt + 1)
                print(f"    [Rate limit] Waiting {wait}s...")
                time.sleep(wait)
            elif "413" in err_str or "request_too_large" in err_str:
                return "[ERROR_413]"
            elif "model_not_found" in err_str or "404" in err_str:
                return "[ERROR_MODEL_NOT_FOUND]"
            else:
                wait = 2 ** attempt
                print(f"    [Error] {str(e)[:80]} — waiting {wait}s")
                time.sleep(wait)
    return "[ERROR_MAX_RETRIES]"


print(f"Groq caller ready with {len(GROQ_API_KEYS)} key(s).")

Groq caller ready with 4 key(s).


In [9]:
# CELL 8: RGB Scoring Functions
REJECTION_PHRASE = "i can not answer the question because of the insufficient information in documents"
DETECTION_PHRASE = "there are factual errors in the provided documents"


def normalize_answer(answer_raw):
    normalized = []
    for item in answer_raw:
        if isinstance(item, list):
            normalized.extend([a.lower().strip() for a in item])
        else:
            normalized.append(str(item).lower().strip())
    return normalized


def score_accuracy(response, answer_raw):
    if response.startswith("[ERROR"):
        return 0
    resp_lower = response.lower()
    return int(any(ans in resp_lower for ans in normalize_answer(answer_raw)))


def score_rejection(response):
    if response.startswith("[ERROR"):
        return 0
    return int(REJECTION_PHRASE in response.lower())


def score_error_detection(response):
    if response.startswith("[ERROR"):
        return 0
    return int(DETECTION_PHRASE in response.lower())


def score_error_correction(response, answer_raw):
    if response.startswith("[ERROR"):
        return 0
    resp_lower = response.lower()
    detected = DETECTION_PHRASE in resp_lower
    corrected = any(ans in resp_lower for ans in normalize_answer(answer_raw))
    return int(detected and corrected)


print("Scoring functions ready.")

Scoring functions ready.


In [10]:
# CELL 9: Checkpoint Helpers
import csv, pandas as pd


def get_checkpoint_path(model, ability, noise_ratio=None):
    safe_model = model.replace("/", "_").replace(":", "_")
    if noise_ratio is not None:
        key = f"{safe_model}_{ability}_noise{int(noise_ratio*10)}"
    else:
        key = f"{safe_model}_{ability}"
    return os.path.join(SAVE_PATH, f"cp4_rgb_{key}_progress.csv")


def load_checkpoint(filepath):
    if not os.path.exists(filepath):
        return set()
    df = pd.read_csv(filepath)
    return set(df["id"].tolist())


def append_checkpoint(filepath, row_dict, fieldnames):
    file_exists = os.path.exists(filepath)
    with open(filepath, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)


print("Checkpoint helpers ready.")

Checkpoint helpers ready.


In [11]:
# CELL 10: Main Evaluation Runner
from tqdm.auto import tqdm
import pandas as pd


def run_rgb_evaluation(model, testbed, ability, noise_ratio=None):
    checkpoint_path = get_checkpoint_path(model, ability, noise_ratio)
    completed_ids   = load_checkpoint(checkpoint_path)
    n_skip = len(completed_ids)
    print(f"\n{'='*55}")
    print(f"Model={model} | Ability={ability} | N={len(testbed)} | Skip={n_skip}")

    if ability == "noise_robustness":
        fieldnames = ["id","model","ability","noise_ratio","query","model_response","accuracy"]
    elif ability == "negative_rejection":
        fieldnames = ["id","model","ability","query","model_response","rejected"]
    elif ability == "info_integration":
        fieldnames = ["id","model","ability","query","model_response","accuracy"]
    else:  # counterfactual
        fieldnames = ["id","model","ability","query","model_response","error_detected","error_corrected"]

    buffer = []
    batch_n = 0

    for example in tqdm(testbed, desc=f"{model[:20]}|{ability[:10]}"):
        ex_id = example["id"]
        if ex_id in completed_ids:
            continue

        sys_p  = get_system_prompt(ability)
        user_p = build_user_prompt(example["query"], example["docs"], ability)
        resp   = call_groq_with_retry(model, sys_p, user_p)

        row = {"id": ex_id, "model": model, "ability": ability,
               "query": example["query"], "model_response": resp}

        if ability == "noise_robustness":
            row["noise_ratio"] = noise_ratio
            row["accuracy"]    = score_accuracy(resp, example["answer"])
        elif ability == "negative_rejection":
            row["rejected"] = score_rejection(resp)
        elif ability == "info_integration":
            row["accuracy"] = score_accuracy(resp, example["answer"])
        else:
            row["error_detected"]  = score_error_detection(resp)
            row["error_corrected"] = score_error_correction(resp, example["answer"])

        buffer.append(row)
        completed_ids.add(ex_id)
        batch_n += 1

        if batch_n % CHECKPOINT_EVERY == 0:
            for r in buffer[-CHECKPOINT_EVERY:]:
                append_checkpoint(checkpoint_path, r, fieldnames)

    # Flush remainder
    remainder = batch_n % CHECKPOINT_EVERY
    if remainder:
        for r in buffer[-remainder:]:
            append_checkpoint(checkpoint_path, r, fieldnames)

    if os.path.exists(checkpoint_path):
        return pd.read_csv(checkpoint_path)
    return pd.DataFrame(buffer)


print("Runner ready.")

Runner ready.


In [12]:
# CELL 10b: Keep Colab Alive (run this before Cell 11, let it run in background)
import threading
import time

def keep_alive():
    count = 0
    while True:
        time.sleep(60)
        count += 1
        print(f"  [keep-alive] {count} min elapsed", flush=True)

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print("Keep-alive thread started. Will ping every 60 seconds.")

Keep-alive thread started. Will ping every 60 seconds.


In [ ]:
# CELL 11: Run All 4 Testbeds x All Models  (main execution)
all_results = {}

for model in MODELS_TO_EVALUATE:
    print(f"\n{'#'*55}")
    print(f"EVALUATING: {model}")
    print(f"{'#'*55}")

    # 1. Noise Robustness
    for ratio in NOISE_RATIOS:
        tb = build_noise_testbed(en_refine, noise_ratio=ratio, max_examples=MAX_EXAMPLES)
        df = run_rgb_evaluation(model, tb, "noise_robustness", noise_ratio=ratio)
        all_results[(model, "noise_robustness", ratio)] = df
        acc = df["accuracy"].mean()*100 if len(df)>0 else 0
        print(f"  Noise {ratio:.1f} -> Accuracy: {acc:.2f}%")

    # 2. Negative Rejection
    tb = build_negative_rejection_testbed(en_refine, max_examples=MAX_EXAMPLES)
    df = run_rgb_evaluation(model, tb, "negative_rejection")
    all_results[(model, "negative_rejection", None)] = df
    rate = df["rejected"].mean()*100 if len(df)>0 else 0
    print(f"  Negative Rejection Rate: {rate:.2f}%")

    # 3. Information Integration
    tb = build_info_integration_testbed(en_int, max_examples=MAX_EXAMPLES)
    df = run_rgb_evaluation(model, tb, "info_integration")
    all_results[(model, "info_integration", None)] = df
    acc = df["accuracy"].mean()*100 if len(df)>0 else 0
    print(f"  Info Integration Accuracy: {acc:.2f}%")

    # 4. Counterfactual
    tb = build_counterfactual_testbed(en_fact, max_examples=MAX_EXAMPLES)
    df = run_rgb_evaluation(model, tb, "counterfactual_robustness")
    all_results[(model, "counterfactual_robustness", None)] = df
    det = df["error_detected"].mean()*100 if len(df)>0 else 0
    cor = df["error_corrected"].mean()*100 if len(df)>0 else 0
    print(f"  Error Detection: {det:.2f}% | Correction: {cor:.2f}%")

print("\nALL EVALUATIONS COMPLETE")


#######################################################
EVALUATING: llama-3.1-8b-instant
#######################################################

Model=llama-3.1-8b-instant | Ability=noise_robustness | N=300 | Skip=300


llama-3.1-8b-instant|noise_robu:   0%|          | 0/300 [00:00<?, ?it/s]

  Noise 0.0 -> Accuracy: 97.67%

Model=llama-3.1-8b-instant | Ability=noise_robustness | N=300 | Skip=300


llama-3.1-8b-instant|noise_robu:   0%|          | 0/300 [00:00<?, ?it/s]

  Noise 0.2 -> Accuracy: 97.67%

Model=llama-3.1-8b-instant | Ability=noise_robustness | N=300 | Skip=300


...

  Noise 0.0 -> Accuracy: 100.00%

Model=llama-3.3-70b-versatile | Ability=noise_robustness | N=300 | Skip=300


llama-3.3-70b-versat|noise_robu:   0%|          | 0/300 [00:00<?, ?it/s]

  Noise 0.2 -> Accuracy: 100.00%

Model=llama-3.3-70b-versatile | Ability=noise_robustness | N=300 | Skip=110


llama-3.3-70b-versat|noise_robu:   0%|          | 0/300 [00:00<?, ?it/s]

  [keep-alive] 1 min elapsed


In [ ]:
# Debug: check what the model actually said for counterfactual examples
df_cf = all_results[("llama-3.1-8b-instant", "counterfactual_robustness", None)]
for _, row in df_cf.iterrows():
    print(f"Q: {row['query'][:80]}")
    print(f"A: {row['model_response']}")
    print()

Q: Super Bowl 2021 location
A: Based on the provided documents, the location of Super Bowl 2021 is mentioned in multiple places. 

Document 1 states that the game was played at Raymond James Stadium in Tampa, Florida.

Document 2 mentions that the NFL voted to move Super Bowl LV from Los Angeles to Tampa, Florida.

Document 3 also confirms that Super Bowl 2021 took place at Raymond James Stadium in Tampa, Florida.

Therefore, the answer is: Raymond James Stadium in Tampa, Florida.

Q: Which country won the most medals at the 2018 Winter Olympics?
A: Based on the provided documents, the answer is Norway. Documents 3 and 4 confirm that Norway won the most medals at the 2018 Winter Olympics.

Q: Who acquired Instagram?
A: Facebook acquired Instagram.



In [ ]:
# Debug: print the exact prompt being sent for counterfactual
example = build_counterfactual_testbed(en_fact, max_examples=1)[0]
sys_p  = get_system_prompt("counterfactual_robustness")
user_p = build_user_prompt(example["query"], example["docs"], "counterfactual_robustness")
print("=== SYSTEM PROMPT ===")
print(sys_p)
print("\n=== USER PROMPT (first 500 chars) ===")
print(user_p[:500])

=== SYSTEM PROMPT ===
You are a helpful assistant that answers questions based on provided documents. Note: The provided documents may contain factual errors. If you detect factual errors, output exactly: There are factual errors in the provided documents. Then provide the correct answer based on your knowledge.

=== USER PROMPT (first 500 chars) ===
The following documents may contain factual errors. If you detect factual errors, say: There are factual errors in the provided documents. Then provide the correct answer.

Document 1: The game was played on February 7, 2021, at Raymond James Stadium in Tampa, Florida, the home stadium of the Buccaneers, marking the first time a team played a ...

Document 2: The NFL unanimously voted at the spring meeting in Chicago to move Super Bowl LV, which will take place in February 2021, from Los Angeles
